# Week 7: API and Web Scraping (part 1)
## Name: Bria Pickel-Fridman
## 10/12/2026

In [1]:
# redo my artist search in homework

import requests

base_url = "https://collectionapi.metmuseum.org"
search_path = "/public/collection/v1.1/search"
search_query = "?q=John%20Singer%20Sargent" # the %20 stops the URL from breaking
artist_or_culture = "&artistOrCulture=true" # flag that limits results to the artist or culture fields
limit = "&limit=500" # sets the amount of responses per page to 500
offset = "&offset=0"

url = f'{base_url}{search_path}{search_query}{artist_or_culture}{limit}{offset}'
search_response = requests.get(url)

print(url)

https://collectionapi.metmuseum.org/public/collection/v1.1/search?q=John%20Singer%20Sargent&artistOrCulture=true&limit=500&offset=0


In [2]:
print(search_response.ok)

True


In [3]:
met_objects = search_response.json()
complete_jss_objects = met_objects["objectIDs"]
print(len(complete_jss_objects))
print ("out of")
print(met_objects['total'])

500
out of
660


In [4]:
limit = "&limit=500" # sets the amount of responses per page to 500
offset = "&offset=500"

url = f'{base_url}{search_path}{search_query}{artist_or_culture}{limit}{offset}'
search_response = requests.get(url)

print(search_response.ok)

complete_jss_objects.extend(search_response.json()["objectIDs"])

print(len(complete_jss_objects))

True
660


In [44]:
from time import sleep # this allows us to use the time finctions related to waiting between operations

object_path = "/public/collection/v1/objects/" # this is the path of the ibjects endpoint in the Met API
# antelope_item=antelope_object_ids[0] # this is the first item in the list of objectIDs above

# url = f"{base_url}{object_path}{antelope_item}"

# response_object = requests.get(url)

# print(response_object.ok)
# print(url)

all_jss_details=[]

for jss_artwork in complete_jss_objects[:10]: # for each of the object IDs I pulled from the MetAPI earlier... limiting to the first 10 as an example
    url = f"{base_url}{object_path}{jss_artwork}"
    response_object=requests.get(url) # making the request to the server
    all_jss_details.append(response_object.json())
    print(response_object.json()['artistDisplayName']) # prints the artist name to be sure our criteria is correct
    print(f"grabbed details from objectID {jss_artwork} with status code {response_object.status_code}")
    sleep(0.5) # wait half a second before moving on to the next step in the rate limiting in the Met's server

print(len(all_jss_details)) # if the item isn't found, and returning a 404, something is off. 

print(all_jss_details[0])

John Singer Sargent
grabbed details from objectID 12025 with status code 200
John Singer Sargent
grabbed details from objectID 12026 with status code 200
John Singer Sargent
grabbed details from objectID 12027 with status code 200
John Singer Sargent
grabbed details from objectID 12028 with status code 200
John Singer Sargent
grabbed details from objectID 12029 with status code 200
John Singer Sargent
grabbed details from objectID 12030 with status code 200
John Singer Sargent
grabbed details from objectID 12031 with status code 200
John Singer Sargent
grabbed details from objectID 12032 with status code 200
John Singer Sargent
grabbed details from objectID 12033 with status code 200
John Singer Sargent
grabbed details from objectID 12034 with status code 200
10
{'objectID': 12025, 'isHighlight': False, 'accessionNumber': '50.130.81m', 'accessionYear': '1950', 'isPublicDomain': True, 'primaryImage': 'https://images.metmuseum.org/CRDImages/ad/original/DT287187.jpg', 'primaryImageSmall':

## NYTimes API examples

In [11]:
# first we import everything e need for the task at hand

import requests # this gives us the ability to request data from remote websites
from time import sleep # this gives us the ability to implement time delay sin our code to meet APi regulations
from getpass import getpass # this is a password protection feature for python notebooks

key = getpass("Paste your API key here!")

Paste your API key here! ········


In [25]:
# this method is setting the parameters as variables then slotting them into the URL with {}

query = "migrant" # searching for the search term 'migrant' in any of the metadata

begin_date = "20260901" # their date format is in ISO format
end_date = "20260930"

sort = "newest"

url = f"https://api.nytimes.com/svc/search/v2/articlesearch.json?q={query}&begin_date={begin_date}&end_date={end_date}&sort={sort}&api-key={key}"
response = requests.get(url)

print(response.status_code)

200


Pulling the actual docuemnts out of the API response:
- first we need to parse the response as JSON, then we need to fin d the keys that contains the article info

In [26]:
parsed_nytimes_response = response.json() # interprets response as Pythin-readable JSON
print(parsed_nytimes_response.keys()) # tells me what keys are in the raw response
print(parsed_nytimes_response['response'].keys()) # tells me what keys are inthe "response parent
nytimes_docs= parsed_nytimes_response['response']['docs'] # tells me what keys are in the "docs" key, which is a child of "response"
print(len(nytimes_docs))

dict_keys(['status', 'copyright', 'response'])
dict_keys(['docs', 'metadata'])
10


In [27]:
print(nytimes_docs[0].keys())

dict_keys(['abstract', 'byline', 'document_type', 'headline', '_id', 'keywords', 'multimedia', 'news_desk', 'print_page', 'print_section', 'pub_date', 'section_name', 'snippet', 'source', 'subsection_name', 'type_of_material', 'uri', 'web_url', 'word_count'])


In [28]:
print(nytimes_docs[0]['headline'])
print(nytimes_docs[0]['headline']['main']) # nested!
print(nytimes_docs[0]['byline']) # shows all of the authors  on the article as a string. 
                                # parsing authors individually is a different story
print(nytimes_docs[0]['abstract'])

{'main': 'Trump Doesn’t Want to Listen to Anyone but Himself', 'kicker': 'Jamelle Bouie', 'print_headline': 'All the Money Belongs to Trump'}
Trump Doesn’t Want to Listen to Anyone but Himself
{'original': 'By Jamelle Bouie'}
The search for fraud in the federal government must always end in the same place.


In [38]:
# this method is setting the parameters as variables then slotting them into the URL with {}

query = '"Donald Trump"' # searching for the search term "Donald Trump" exact order in the API

begin_date = "19000101" # their date format is in ISO format
end_date = "20260930"

sort = "oldest" # looking for the first instance of his name ever in the Times

url = f"https://api.nytimes.com/svc/search/v2/articlesearch.json?q={query}&begin_date={begin_date}&end_date={end_date}&sort={sort}&api-key={key}"
response = requests.get(url)

print(response.status_code)

200


In [39]:
parsed_nytimes_response_2 = response.json() # interprets response as Pythin-readable JSON
print(parsed_nytimes_response_2.keys()) # tells me what keys are in the raw response
print(parsed_nytimes_response_2['response'].keys()) # tells me what keys are inthe "response parent
nytimes_docs_2= parsed_nytimes_response_2['response']['docs'] # tells me what keys are in the "docs" key, which is a child of "response"
print(len(nytimes_docs_2))

dict_keys(['status', 'copyright', 'response'])
dict_keys(['docs', 'metadata'])
10


In [40]:
print(nytimes_docs_2[0].keys())

dict_keys(['abstract', 'byline', 'document_type', 'headline', '_id', 'keywords', 'multimedia', 'news_desk', 'print_page', 'print_section', 'pub_date', 'section_name', 'snippet', 'source', 'subsection_name', 'type_of_material', 'uri', 'web_url', 'word_count'])


In [41]:
print(nytimes_docs_2[0]['headline'])
print(nytimes_docs_2[0]['headline']['main']) # nested!
print(nytimes_docs_2[0]['byline']) # shows all of the authors  on the article as a string. 
                                # parsing authors individually is a different story
print(nytimes_docs_2[0]['abstract'])
print(nytimes_docs_2[0]['pub_date'])

{'main': 'A Builder Looks Back—and Moves Forward', 'kicker': '', 'print_headline': 'A Builder Looks Back—and Moves Forward'}
A Builder Looks Back—and Moves Forward
{'original': 'By Alden Whitman'}

1973-01-28T05:00:00Z


In [45]:
# check professors class notes later

KeyError: 'metadata'

## Web Scraping

In [46]:
import csv
import json
import os #operating system library, lets us do things like making new folders
import requests
from time import sleep
from bs4 import BeautifulSoup # the scraping library!

In [47]:
# below is an example of HTML that we can traverse using the BeautifulSoup functions

html = """
<html>
  <body>
    <h1>Adeline Ravoux</h1>
    <p class="artist">Vincent van Gogh</p>
    <p class="date">1890</p>
    <a href="/art/1958.31">Artwork page</a>
  </body>
</html>
"""

soup = BeautifulSoup(html, "html.parser")  # html.parser is the built-in parser that comes with python
print(dir(soup))

['ASCII_SPACES', 'DEFAULT_BUILDER_FEATURES', 'EMPTY_ELEMENT_EVENT', 'END_ELEMENT_EVENT', 'MAIN_CONTENT_STRING_TYPES', 'ROOT_TAG_NAME', 'START_ELEMENT_EVENT', 'STRING_ELEMENT_EVENT', '_TreeTraversalEvent', '__annotations__', '__bool__', '__call__', '__class__', '__contains__', '__copy__', '__deepcopy__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setitem__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__unicode__', '__weakref__', '_all_strings', '_decode_markup', '_event_stream', '_feed', '_find_all', '_find_one', '_format_tag', '_indent_string', '_insert', '_is_xml', '_lastRecursiveChild', '_last_descendant', '_linkage_fixer', '_markup_is_url', '_markup_resem

In [48]:
title = soup.find("h1") # finds the first element that uses the h1 tag
print(title)
print(title.get_text()) # removes the tags and just shows you the text

<h1>Adeline Ravoux</h1>
Adeline Ravoux


In [49]:
paragraph_elements = soup.find_all("p")
print(paragraph_elements)

[<p class="artist">Vincent van Gogh</p>, <p class="date">1890</p>]


In [51]:
artist=soup.find("p", class_="artist") # the underscrore was added to class becuase class is a reserved word in python
print(artist.get_text())

Vincent van Gogh


In [56]:
date = soup.find("p", class_="date")
print(date.get_text())

1890


- right click>inspect (for chrome or firefox)
- view>Developer>Inspect Elements (for chrome)


# understanding "Robots" permissions

We can view the permissions rthat a website automatically grants bto do things like automatically access data with a "bot" by looking at its `robots.txt` file. it always lives at the robot.txt file at the root of the directory
robots.txt

In [ ]:
headers = {"User-Agent": "INFO664-class-project (educational)"}   # you can call this whatever you want, it shows up on the scraped website's logs as who you are (ie not a Google/AI bot or web browser)

os.makedirs("scraped_pages", exist_ok=True)  # make a new folder to hold our scraped website HTML data. exist_ok = True means, if the directory already exists, do nothing and don't return an error

def fetch_page(url, filename):  # declaring a new function, which takes a URL to be scraped, and a destination filename, as arguments
    if os.path.exists(filename):  # check to see if file already exists, if so, don't download it from the web, load it instead
        with open(filename, "r", encoding="utf-8") as file:
            return file.read()  # in this case, return provides the read file data, and escapes from the function, ignoring everything below

    response = requests.get(url, headers=headers) # grab the file, using our pre-defined headers as the header data

    if not response.ok: # if the response reports an error
        print(url, "failed with status code", response.status_code) # tell us why it failed
        return None # return a "None" object, since there's nothing usable in the response, and escape from this function
    with open(filename, "w",  encoding="utf-8") as file:
        file.write(response.text)

    sleep(10) # wait 10 seconds, since our robots.txt file asks for a 10 second Crawl-delay
    return response.text